In [3]:
import wrds
import pandas as pd

db = wrds.Connection(wrds_username='coleginter')

Loading library list...
Done


In [4]:
db.raw_sql("select 1 as one")

,one
0,1


In [6]:
query = """
SELECT gvkey, conm
FROM comp.company
WHERE UPPER(conm) LIKE '%%DREXEL%%'
LIMIT 50
"""
df = db.raw_sql(query)
df

,gvkey,conm
0,004075,DREXEL INDUSTRIES INC-PA
1,004076,DREXEL ENTERPRISES INC-UTAH


In [10]:
missing = [
    "AUBREY G. LANSTON & CO., INC.",
    "BARTOW LEEDS & CO.",
    "C.F. CHILDS & CO., INC.",
    "CONTINENTAL ILL.",
    "D.W. RICH & CO., INC",
    "DREXEL BURNHAM LAMBERT",
    "MALON S. ANDRUS INC.",
    "SECURITIES GROUPS",
    "THE FIRST BOSTON CORPORATION",
    "WM. E. POLLOCK GOV''T SECURITIES,INC",
    "SECOND DISTRICT SECURITIES CO., INC",
    "F.I. DUPONT & CO BECKER",
    "FIRST PENNCO SEC. INC.",
    "WHITE, WELD & CO INC.",
    "WEEDEN & CO., INC.",
    "KLEINWORT BENSON GOV''T SEC. INC.",
    "BEAR, STEARNS & CO., INC.",
    "N.Y. HANSEATIC CORP.",
    "L.F. ROTHSCHILD,UNTERBERG,TOWBIN",
    "BROPHY,GESTAL, KNIGHT AND CO., L.P.",
    "CRT GOVERNMENT SECURITIES,INC.",
    "THE NIKKO SECURITIES CO. INT''L",
    "SANWA-BGK SECURITIES CO., L.P.",
    "S.G. WARBURG & CO., INC.",
    "WERTHEIM SCHRODER & CO., INC.",
    "YAMAICHI INT''L (AMERICA), INC",
    "SBC GOVERNMENT SECURITIES, INC.",
    "EASTBRIDGE CAPITAL INC.",
    "NATIONSBANK OF NORTH CAROLINA, N.A.",
    "NATIONSBANC CAPITAL MARKETS, INC.",
    "SANWA SECURITIES (USA) CO., L.",
    "SBC CAPITAL MARKETS INC.",
    "SBC WARBURG INC.",
    "SBC WARBURG DILLON READ INC.",
    "NATIONSBANC MONTGOMERY SECUR.",
    "WARBURG DILLON READ LLC.",
    "ABN AMRO INCORPORATED",
    "SG COWEN SECURITIES CORP.",
    "RBS SECURITIES INC.",
    "SG AMERICAS SECURITIES, LLC"
]

# build OR clause safely
ors = " OR ".join([f"UPPER(conm) LIKE '%%{name.upper()}%%'" for name in missing])

query = f"""
SELECT gvkey, conm
FROM comp.company
WHERE {ors}
ORDER BY conm
"""
hits = db.raw_sql(query)
hits

,gvkey,conm


In [11]:
import re
import pandas as pd

def norm(s: str) -> str:
    # keep letters/numbers/spaces only
    return re.sub(r"[^A-Z0-9 ]+", " ", s.upper()).split()

def token_where_sql(col_sql: str, name: str) -> str:
    # require ALL tokens to appear
    toks = norm(name)
    # keep tokens that are informative
    toks = [t for t in toks if t not in {"CO", "INC", "LLC", "L", "LP", "CORP", "CORPORATION", "THE", "OF", "AND"}]
    if not toks:
        toks = norm(name)
    # Postgres regex-replace for normalization inside SQL
    norm_col = f"regexp_replace(upper({col_sql}), '[^A-Z0-9 ]', ' ', 'g')"
    return " AND ".join([f"{norm_col} LIKE '%{t}%'" for t in toks])

def find_gvkey_candidates(db, dealer_name: str, limit: int = 25) -> pd.DataFrame:
    where_names = token_where_sql("conm", dealer_name)   # comp.names uses 'conm'
    where_comp  = token_where_sql("conm", dealer_name)   # comp.company uses 'conm' too

    q = f"""
    (
      SELECT 'names' AS src, gvkey, conm, nameendt, namestdt
      FROM comp.names
      WHERE {where_names}
      ORDER BY nameendt DESC NULLS LAST
      LIMIT {limit}
    )
    UNION ALL
    (
      SELECT 'company' AS src, gvkey, conm, NULL::date AS nameendt, NULL::date AS namestdt
      FROM comp.company
      WHERE {where_comp}
      LIMIT {limit}
    );
    """
    return db.raw_sql(q)

# Example: Drexel Burnham Lambert
cand = find_gvkey_candidates(db, "DREXEL BURNHAM LAMBERT", limit=50)
cand.head(25)

KeyError: 0

In [13]:
import wrds
db = wrds.Connection(wrds_username='coleginter')

pat = "%DREXEL%"
q = """
SELECT gvkey, conm
FROM comp.company
WHERE upper(conm) LIKE upper(%(pat)s)
LIMIT 50
"""
df = db.raw_sql(q, params={"pat": pat})
df

Loading library list...
Done


,gvkey,conm
0,004075,DREXEL INDUSTRIES INC-PA
1,004076,DREXEL ENTERPRISES INC-UTAH


In [15]:
import re
import pandas as pd

missing = missing = [
    "AUBREY G. LANSTON & CO., INC.",
    "BARTOW LEEDS & CO.",
    "C.F. CHILDS & CO., INC.",
    "CONTINENTAL ILL.",
    "D.W. RICH & CO., INC",
    "DREXEL BURNHAM LAMBERT",
    "MALON S. ANDRUS INC.",
    "SECURITIES GROUPS",
    "THE FIRST BOSTON CORPORATION",
    "WM. E. POLLOCK GOV''T SECURITIES,INC",
    "SECOND DISTRICT SECURITIES CO., INC",
    "F.I. DUPONT & CO BECKER",
    "FIRST PENNCO SEC. INC.",
    "WHITE, WELD & CO INC.",
    "WEEDEN & CO., INC.",
    "KLEINWORT BENSON GOV''T SEC. INC.",
    "BEAR, STEARNS & CO., INC.",
    "N.Y. HANSEATIC CORP.",
    "L.F. ROTHSCHILD,UNTERBERG,TOWBIN",
    "BROPHY,GESTAL, KNIGHT AND CO., L.P.",
    "CRT GOVERNMENT SECURITIES,INC.",
    "THE NIKKO SECURITIES CO. INT''L",
    "SANWA-BGK SECURITIES CO., L.P.",
    "S.G. WARBURG & CO., INC.",
    "WERTHEIM SCHRODER & CO., INC.",
    "YAMAICHI INT''L (AMERICA), INC",
    "SBC GOVERNMENT SECURITIES, INC.",
    "EASTBRIDGE CAPITAL INC.",
    "NATIONSBANK OF NORTH CAROLINA, N.A.",
    "NATIONSBANC CAPITAL MARKETS, INC.",
    "SANWA SECURITIES (USA) CO., L.",
    "SBC CAPITAL MARKETS INC.",
    "SBC WARBURG INC.",
    "SBC WARBURG DILLON READ INC.",
    "NATIONSBANC MONTGOMERY SECUR.",
    "WARBURG DILLON READ LLC.",
    "ABN AMRO INCORPORATED",
    "SG COWEN SECURITIES CORP.",
    "RBS SECURITIES INC.",
    "SG AMERICAS SECURITIES, LLC"
]

STOP = {
    "CO","INC","LLC","LP","L","CORP","CORPORATION","THE","OF","AND",
    "SECURITIES","SEC","GOVT","GOV","COMPANY"
}

def norm_tokens(name: str, max_tokens: int = 5):
    toks = re.findall(r"[A-Z0-9]+", name.upper())
    toks = [t for t in toks if t not in STOP]
    return toks[:max_tokens] if toks else [name.upper()]

def compustat_candidates(db, dealer: str, limit: int = 25) -> pd.DataFrame:
    toks = norm_tokens(dealer)

    # Require ALL tokens to appear somewhere in the name
    clauses = " AND ".join([f"UPPER(conm) LIKE %(p{i})s" for i in range(len(toks))])
    params = {f"p{i}": f"%{t}%" for i, t in enumerate(toks)}

    q_names = f"""
        SELECT
            'names' AS src,
            gvkey,
            conm,
            namedt,
            nameendt
        FROM comp.names
        WHERE {clauses}
        ORDER BY nameendt DESC NULLS LAST
        LIMIT {limit}
    """

    q_company = f"""
        SELECT
            'company' AS src,
            gvkey,
            conm,
            NULL::date AS namedt,
            NULL::date AS nameendt
        FROM comp.company
        WHERE {clauses}
        LIMIT {limit}
    """
    a = db.raw_sql(q_names, params=params)
    b = db.raw_sql(q_company, params=params)

    out = pd.concat([a, b], ignore_index=True)
    out.insert(0, "dealer_query", dealer)
    return out

# ---- Run for multiple dealers (start small) ----
# Example: just 5 first
hits = pd.concat([compustat_candidates(db, d, limit=15) for d in missing[:5]], ignore_index=True)
hits

ProgrammingError: (psycopg2.errors.UndefinedColumn) column "namedt" does not exist
LINE 6:             namedt,
                    ^

[SQL: 
        SELECT
            'names' AS src,
            gvkey,
            conm,
            namedt,
            nameendt
        FROM comp.names
        WHERE UPPER(conm) LIKE %(p0)s AND UPPER(conm) LIKE %(p1)s AND UPPER(conm) LIKE %(p2)s
        ORDER BY nameendt DESC NULLS LAST
        LIMIT 15
    ]
[parameters: {'p0': '%AUBREY%', 'p1': '%G%', 'p2': '%LANSTON%'}]
(Background on this error at: https://sqlalche.me/e/20/f405)